In [ ]:
# In[1]:

import pandas as pd

# Read CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_err['timestamp'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Incident window (UTC)
inc_start = pd.to_datetime("2024-01-24 02:06:40", utc=True)
inc_end = pd.to_datetime("2024-01-24 02:36:40", utc=True)

# 1) metric.csv summary: unique (cmdb_id, kpi_name) pairs with count and global P95 (full series)
metric_grp = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(count='size', p95=lambda x: x.quantile(0.95))
)
metric_grp['p95'] = metric_grp['p95'].round(6)
metric_summary = metric_grp.sort_values('count', ascending=False).head(20)

# metric raw samples inside incident window (up to 5 rows, all columns)
metric_samples = df_metric[(df_metric['timestamp'] >= inc_start) & (df_metric['timestamp'] <= inc_end)].head(5)

# 2) trace.csv summary: unique (cmdb_id, trace_name) pairs with count and global P95
trace_grp = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .agg(count='size', p95=lambda x: x.quantile(0.95))
)
trace_grp['p95'] = trace_grp['p95'].round(6)
trace_summary = trace_grp.sort_values('count', ascending=False).head(20)

# trace raw samples inside incident window (up to 5 rows)
trace_samples = df_trace[(df_trace['timestamp'] >= inc_start) & (df_trace['timestamp'] <= inc_end)].head(5)

# 3) log.csv summary: unique (cmdb_id, log_name) pairs with count and global P95
log_grp = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)['value']
    .agg(count='size', p95=lambda x: x.quantile(0.95))
)
log_grp['p95'] = log_grp['p95'].round(6)
log_summary = log_grp.sort_values('count', ascending=False).head(20)

# log raw samples inside incident window (up to 5 rows)
log_samples = df_log[(df_log['timestamp'] >= inc_start) & (df_log['timestamp'] <= inc_end)].head(5)

# 4) error_logs.csv: unique cmdb_id values and count of error log records per cmdb_id
err_grp = (
    df_err
    .groupby('cmdb_id', as_index=False)
    .agg(count=('cmdb_id','size'))
    .sort_values('count', ascending=False)
)
error_summary = err_grp.head(20)

# error log raw samples inside incident window (up to 5 rows)
error_samples = df_err[(df_err['timestamp'] >= inc_start) & (df_err['timestamp'] <= inc_end)].head(5)

# Compact labeled outputs (DataFrames). Display as variables (IPython will render them).
metric_summary, metric_samples, trace_summary, trace_samples, log_summary, log_samples, error_summary, error_samples

```
Out[1]:
```
```python
summary = (
    "Telemetry load summary (incident window: 2024-01-24 02:06:40 to 2024-01-24 02:36:40 UTC):\n\n"
    "1) metric.csv (global P95 computed over full series before filtering):\n"
    "- Many (cmdb_id, kpi_name) pairs have 25 data points each in the top results.\n"
    "- Notable high P95 values appear for disk I/O and memory KPIs on admin services (e.g., ts-admin-basic-info-service diskio and mem P95s are large).\n"
    "- Sample metric rows inside the incident window (up to 5) show ts-admin-basic-info-service metrics at 2024-01-24 02:09:00 UTC.\n\n"
    "2) trace.csv (global P95 over full series):\n"
    "- Multiple trace pairs present (counts generally 25). Many root->service traces recorded.\n"
    "- Some traces show elevated P95s (example: traces to ts-auth-service had a higher duration_p95 ~1.13s in the summary).\n"
    "- Sample trace rows inside the incident window (up to 5) include root traces to ts-admin-basic-info-service and ts-admin-travel-service at 2024-01-24 02:09:00 UTC.\n\n"
    "3) log.csv (global P95 over full series):\n"
    "- log.error_count is zero for the top services in the summary (no aggregated error counts in those series).\n"
    "- log.row_count P95s are high for several services (e.g., ts-travel-service ~521, ts-travel2-service ~528.8, ts-auth-service ~381.6), indicating high request/row volumes.\n"
    "- Sample log rows inside the incident window (up to 5) show log.row_count entries (e.g., ts-admin-basic-info-service and ts-admin-travel-service at 02:09:00 UTC).\n\n"
    "4) error_logs.csv:\n"
    "- No error log records found in the dataset (empty result for error log rows inside the incident window and overall grouping returned empty).\n\n"
    "Overall interpretation / pointers:\n"
    "- There are no recorded error logs and trace error_rate entries in the sampled data were zero, so explicit error logs are not present.\n"
    "- Telemetry indicates elevated disk I/O and memory P95s on admin services and some elevated trace latency to the auth service; these are the primary anomalies to investigate further as potential causes.\n"
    "- Recommended next steps: examine the services with high diskio/memory P95 (e.g., ts-admin-basic-info-service), investigate trace spans to ts-auth-service for latency spikes, and correlate with any upstream load (high log.row_count) during the incident window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count           p95
337           ts-voucher-service      socket     25  4.000000e+00
0    ts-admin-basic-info-service         cpu     25  1.986964e+00
1    ts-admin-basic-info-service      diskio     25  5.865918e+04
2    ts-admin-basic-info-service       error     25  0.000000e+00
3    ts-admin-basic-info-service  latency-50     25  1.779500e-02
4    ts-admin-basic-info-service  latency-90     25  2.403100e-02
5    ts-admin-basic-info-service         mem     25  2.317392e+08
6    ts-admin-basic-info-service      socket     25  5.996667e+00
7    ts-admin-basic-info-service    workload     25  7.315670e-01
8         ts-admin-order-service         cpu     25  1.113010e-01
9         ts-admin-order-service      diskio     25  5.318266e+04
10        ts-admin-order-service         mem     25  1.965201e+08
11        ts-admin-order-service      socket     25  3.396667e+00
12        ts-admin-route-service         cpu     25  1.147760e-01
13        ts-admin-route-service      diskio     25  4.595374e+04
14        ts-admin-route-service         mem     25  1.955913e+08
15        ts-admin-route-service      socket     25  3.453333e+00
16       ts-admin-travel-service         cpu     25  4.733180e-01
298            ts-travel-service    workload     25  1.828800e+00
299             ts-travel2-mongo         cpu     25  5.187920e-01,                   timestamp                      cmdb_id    kpi_name         value
0 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service         cpu      0.327686
1 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service      diskio  47099.901921
2 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service       error      0.000000
3 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service  latency-50      0.017690
4 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service  latency-90      0.023843,                 cmdb_id                                         trace_name  count         p95
399  ts-travel2-service              trace.to_ts-travel2-service.row_count     25  642.800000
0                  root  trace.to_ts-admin-basic-info-service.duration_...     25    0.013252
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     25    0.021458
2                  root    trace.to_ts-admin-basic-info-service.error_rate     25    0.000000
3                  root     trace.to_ts-admin-basic-info-service.row_count     25   21.000000
4                  root     trace.to_ts-admin-travel-service.duration_mean     25    0.152295
5                  root      trace.to_ts-admin-travel-service.duration_p95     25    0.244858
6                  root        trace.to_ts-admin-travel-service.error_rate     25    0.000000
7                  root         trace.to_ts-admin-travel-service.row_count     25   11.000000
8                  root        trace.to_ts-assurance-service.duration_mean     25    0.004583
9                  root         trace.to_ts-assurance-service.duration_p95     25    0.008618
10                 root           trace.to_ts-assurance-service.error_rate     25    0.000000
11                 root            trace.to_ts-assurance-service.row_count     25   10.800000
12                 root             trace.to_ts-auth-service.duration_mean     25    0.375699
13                 root              trace.to_ts-auth-service.duration_p95     25    1.131621
14                 root                trace.to_ts-auth-service.error_rate     25    0.000000
360   ts-travel-service       trace.to_ts-ticketinfo-service.duration_mean     25    0.011205
361   ts-travel-service        trace.to_ts-ticketinfo-service.duration_p95     25    0.016609
362   ts-travel-service          trace.to_ts-ticketinfo-service.error_rate     25    0.000000
363   ts-travel-service           trace.to_ts-ticketinfo-service.row_count     25   54.000000,                   timestamp cmdb_id                                         trace_name     value
0 2024-01-24 02:09:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_...  0.012644
1 2024-01-24 02:09:00+00:00    root  trace.to_ts-admin-basic-info-service.duration_p95  0.014601
2 2024-01-24 02:09:00+00:00    root    trace.to_ts-admin-basic-info-service.error_rate  0.000000
3 2024-01-24 02:09:00+00:00    root     trace.to_ts-admin-basic-info-service.row_count  6.000000
4 2024-01-24 02:09:00+00:00    root     trace.to_ts-admin-travel-service.duration_mean  0.141808,                         cmdb_id         log_name  count    p95
0   ts-admin-basic-info-service  log.error_count     25    0.0
1   ts-admin-basic-info-service    log.row_count     25   31.8
2       ts-admin-travel-service  log.error_count     25    0.0
3       ts-admin-travel-service    log.row_count     25   33.0
7          ts-assurance-service    log.row_count     25   10.8
6          ts-assurance-service  log.error_count     25    0.0
12             ts-basic-service  log.error_count     25    0.0
13             ts-basic-service    log.row_count     25  225.2
11              ts-auth-service    log.row_count     25  381.6
10              ts-auth-service  log.error_count     25    0.0
27          ts-food-map-service    log.row_count     25   22.0
26          ts-food-map-service  log.error_count     25    0.0
21           ts-consign-service    log.row_count     25   72.8
20           ts-consign-service  log.error_count     25    0.0
68            ts-travel-service  log.error_count     25    0.0
69            ts-travel-service    log.row_count     25  521.0
72           ts-travel2-service  log.error_count     25    0.0
73           ts-travel2-service    log.row_count     25  528.8
45             ts-order-service    log.row_count     25   50.6
44             ts-order-service  log.error_count     25    0.0,                   timestamp                      cmdb_id         log_name  value
0 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service  log.error_count      0
1 2024-01-24 02:09:00+00:00  ts-admin-basic-info-service    log.row_count     10
2 2024-01-24 02:09:00+00:00      ts-admin-travel-service  log.error_count      0
3 2024-01-24 02:09:00+00:00      ts-admin-travel-service    log.row_count     12
4 2024-01-24 02:09:00+00:00           ts-assurance-mongo  log.error_count      0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message, error]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reload metric.csv to ensure freshest state and parse timestamps (UTC)
df_metric = pd.read_csv("metric.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
inc_start = pd.to_datetime("2024-01-24 02:06:40", utc=True)
inc_end = pd.to_datetime("2024-01-24 02:36:40", utc=True)

# 1) Compute global P95 and P15 for each (cmdb_id, kpi_name) over the full metric series
thresholds_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P15=lambda x: x.quantile(0.15))
)
# keep thresholds in kernel variable thresholds_df

# 2) Filter metric.csv to the incident window
df_window = df_metric[(df_metric['timestamp'] >= inc_start) & (df_metric['timestamp'] <= inc_end)].copy()

# 3) Identify anomalous points within window using global thresholds (thresholds computed above)
# Merge thresholds into the windowed data
window_with_thresh = df_window.merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')

# Mark spike and drop anomalies
window_with_thresh['is_spike'] = window_with_thresh['value'] > window_with_thresh['global_P95']
window_with_thresh['is_drop'] = window_with_thresh['value'] <= window_with_thresh['global_P15']
window_with_thresh['is_anom'] = window_with_thresh['is_spike'] | window_with_thresh['is_drop']

# Extract anomalous rows
anomalous_rows = window_with_thresh[window_with_thresh['is_anom']].copy()

# Compute per-pair window stats (max/min in the window, even for pairs with anomalies)
window_stats = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(max_value_in_window='max', min_value_in_window='min')
)

# Compute anomalous counts and earliest anomaly time per pair
anom_group = (
    anomalous_rows
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        anomalous_point_count_in_window=('is_anom', 'size'),
        earliest_anomaly_timestamp=('timestamp', 'min'),
        has_spike=('is_spike', 'any'),
        has_drop=('is_drop', 'any')
    )
)

# Merge thresholds and window stats into the anomaly summary
anom_summary = (
    anom_group
    .merge(thresholds_df, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# Compute severity scores:
# - spike_severity = max_value_in_window / global_P95  (only meaningful if has_spike and global_P95>0)
# - drop_severity  = global_P15 / min_value_in_window  (only meaningful if has_drop and min_value_in_window>0)
def compute_severity(row):
    spike_sev = np.nan
    drop_sev = np.nan
    if row['has_spike'] and row['global_P95'] > 0:
        spike_sev = row['max_value_in_window'] / row['global_P95']
    if row['has_drop'] and row['min_value_in_window'] > 0:
        drop_sev = row['global_P15'] / row['min_value_in_window']
    # choose the maximum non-NaN severity (if both exist)
    if np.isnan(spike_sev) and np.isnan(drop_sev):
        return np.nan
    if np.isnan(spike_sev):
        return drop_sev
    if np.isnan(drop_sev):
        return spike_sev
    return max(spike_sev, drop_sev)

anom_summary['severity_score'] = anom_summary.apply(compute_severity, axis=1)

# Keep and format requested columns
final_cols = [
    'cmdb_id', 'kpi_name', 'global_P95', 'global_P15',
    'anomalous_point_count_in_window', 'earliest_anomaly_timestamp',
    'max_value_in_window', 'min_value_in_window', 'severity_score'
]
anom_summary = anom_summary[final_cols].copy()

# Round numeric thresholds/severity for compact display
for c in ['global_P95', 'global_P15', 'max_value_in_window', 'min_value_in_window', 'severity_score']:
    anom_summary[c] = pd.to_numeric(anom_summary[c], errors='coerce').round(6)

# 4) Sort and limit to top 20 by anomalous count desc, then severity desc
anom_summary_sorted = anom_summary.sort_values(
    by=['anomalous_point_count_in_window', 'severity_score'],
    ascending=[False, False]
).head(20)

# Up to 10 raw anomalous rows for inspection (compact columns)
anomalous_raw_samples = anomalous_rows.sort_values('timestamp')[['timestamp', 'cmdb_id', 'kpi_name', 'value']].head(10)

# Display compact results (store full tables in kernel variables)
thresholds_df, df_window.shape, anom_summary_sorted, anomalous_raw_samples

```
Out[2]:
```
summary = (
    "Anomaly summary for metric.csv using global thresholds (full-series P95/P15), window 2024-01-24 02:06:40–02:36:40 UTC:\n\n"
    "- Thresholds were computed and stored for every (cmdb_id, kpi_name) pair. The window filter produced 8,450 metric rows.\n\n"
    "- Key findings (compact):\n"
    "  • Many Mongo and service socket KPIs (e.g., ts-assurance-mongo, ts-auth-mongo, ts-avatar-service, ts-config-mongo, ts-consign-mongo, ts-route-mongo, etc.) show anomalous activity across the entire window (anomalous_point_count_in_window = 25, earliest anomaly 2024-01-24 02:09:00 UTC). Severity scores for those socket entries are 1.0 in the summary (the observed values equal their global thresholds).\n"
    "  • Some memory and disk I/O metrics are notable in the window. Examples from sampled anomalous rows: ts-rebook-service memory ~1.866e8, ts-route-mongo diskio ~2.49476e5 and mem ~6.75103e7. These appear as anomalous relative to their global thresholds.\n"
    "  • The earliest anomalous timestamps are clustered at ~2024-01-24 02:09:00 UTC across many pairs, indicating a simultaneous change across services at that time.\n\n"
    "- Raw anomalous sample rows (for inspection) include metrics such as ts-admin-basic-info-service cpu ~0.3277 and several high mem/diskio/socket entries at 02:09:00 UTC.\n\n"
    "- Interpretation and next steps:\n"
    "  • The most widespread anomalies are on socket KPIs across many services/datastores — investigate networking/socket behavior and any deployment or configuration changes around 02:09 UTC.\n"
    "  • Also investigate services with high mem/diskio values in the window (e.g., ts-rebook-service mem, ts-route-mongo diskio/mem) as potential resource pressure contributors.\n"
    "  • Correlate these metric anomalies with traces and logs around 02:09 UTC to find causal traces or client load spikes.\n\n"
    "If you want, I can display the top anomalous pairs table or more raw anomalous rows for deeper inspection."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name    global_P95    global_P15
0    ts-admin-basic-info-service         cpu  1.986964e+00  3.343940e-01
1    ts-admin-basic-info-service      diskio  5.865918e+04  4.580722e+04
2    ts-admin-basic-info-service       error  0.000000e+00  0.000000e+00
3    ts-admin-basic-info-service  latency-50  1.779493e-02  1.750000e-02
4    ts-admin-basic-info-service  latency-90  2.403088e-02  2.350000e-02
..                           ...         ...           ...           ...
333             ts-voucher-mysql      socket  2.000000e+00  2.000000e+00
334           ts-voucher-service         cpu  3.377242e-02  3.231058e-02
335           ts-voucher-service      diskio  0.000000e+00  0.000000e+00
336           ts-voucher-service         mem  3.799450e+07  3.799207e+07
337           ts-voucher-service      socket  4.000000e+00  4.000000e+00

[338 rows x 4 columns], (8450, 4),                      cmdb_id kpi_name   global_P95   global_P15  anomalous_point_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  severity_score
31        ts-assurance-mongo   socket          3.0          3.0                               25  2024-01-24 02:09:00+00:00                  3.0                  3.0             1.0
42             ts-auth-mongo   socket          6.0          6.0                               25  2024-01-24 02:09:00+00:00                  6.0                  6.0             1.0
53         ts-avatar-service   socket          5.0          5.0                               25  2024-01-24 02:09:00+00:00                  5.0                  5.0             1.0
68           ts-config-mongo   socket          4.0          4.0                               25  2024-01-24 02:09:00+00:00                  4.0                  4.0             1.0
79          ts-consign-mongo   socket          5.0          5.0                               25  2024-01-24 02:09:00+00:00                  5.0                  5.0             1.0
101        ts-contacts-mongo   socket          3.0          3.0                               25  2024-01-24 02:09:00+00:00                  3.0                  3.0             1.0
113        ts-food-map-mongo   socket          5.0          5.0                               25  2024-01-24 02:09:00+00:00                  5.0                  5.0             1.0
124            ts-food-mongo   socket          3.0          3.0                               25  2024-01-24 02:09:00+00:00                  3.0                  3.0             1.0
127            ts-food-mysql      mem  600006656.0  600006656.0                               25  2024-01-24 02:09:00+00:00          600006656.0          600006656.0             1.0
128            ts-food-mysql   socket          2.0          2.0                               25  2024-01-24 02:09:00+00:00                  2.0                  2.0             1.0
139  ts-inside-payment-mongo   socket          3.0          3.0                               25  2024-01-24 02:09:00+00:00                  3.0                  3.0             1.0
149          ts-news-service   socket          1.0          1.0                               25  2024-01-24 02:09:00+00:00                  1.0                  1.0             1.0
153    ts-notification-mongo   socket          2.0          2.0                               25  2024-01-24 02:09:00+00:00                  2.0                  2.0             1.0
161           ts-order-mongo   socket          5.0          5.0                               25  2024-01-24 02:09:00+00:00                  5.0                  5.0             1.0
165     ts-order-other-mongo   socket          4.0          4.0                               25  2024-01-24 02:09:00+00:00                  4.0                  4.0             1.0
183         ts-payment-mongo   socket          3.0          3.0                               25  2024-01-24 02:09:00+00:00                  3.0                  3.0             1.0
192        ts-preserve-mongo   socket          2.0          2.0                               25  2024-01-24 02:09:00+00:00                  2.0                  2.0             1.0
196  ts-preserve-other-mongo   socket          2.0          2.0                               25  2024-01-24 02:09:00+00:00                  2.0                  2.0             1.0
208           ts-price-mongo   socket          4.0          4.0                               25  2024-01-24 02:09:00+00:00                  4.0                  4.0             1.0
223           ts-route-mongo   socket          5.0          5.0                               25  2024-01-24 02:09:00+00:00                  5.0                  5.0             1.0,                     timestamp                      cmdb_id kpi_name         value
0   2024-01-24 02:09:00+00:00  ts-admin-basic-info-service      cpu  3.276864e-01
218 2024-01-24 02:09:00+00:00            ts-rebook-service      mem  1.866097e+08
219 2024-01-24 02:09:00+00:00            ts-rebook-service   socket  3.000000e+00
220 2024-01-24 02:09:00+00:00               ts-route-mongo      cpu  5.626171e-01
221 2024-01-24 02:09:00+00:00               ts-route-mongo   diskio  2.496476e+05
222 2024-01-24 02:09:00+00:00               ts-route-mongo      mem  6.751027e+07
223 2024-01-24 02:09:00+00:00               ts-route-mongo   socket  5.000000e+00
226 2024-01-24 02:09:00+00:00        ts-route-plan-service      mem  1.909719e+08
227 2024-01-24 02:09:00+00:00        ts-route-plan-service   socket  3.000000e+00
229 2024-01-24 02:09:00+00:00             ts-route-service   diskio  4.052248e+04)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reload files and parse timestamps to ensure fresh state
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Candidate services and incident window
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
inc_start = pd.to_datetime("2024-01-24 02:06:40", utc=True)
inc_end = pd.to_datetime("2024-01-24 02:36:40", utc=True)

# ---------------------------
# 1) metric.csv for candidates
# ---------------------------
# Compute global thresholds (P95, P15) across full metric series for candidate cmdb_ids
metric_thresholds = (
    df_metric[df_metric['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P15=lambda x: x.quantile(0.15))
)

# Filter metric rows to incident window for candidate services
df_metric_window = df_metric[
    (df_metric['timestamp'] >= inc_start) & (df_metric['timestamp'] <= inc_end) & 
    (df_metric['cmdb_id'].isin(candidates))
].copy()

# Merge thresholds into windowed metrics
metric_win_with_thresh = df_metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies
metric_win_with_thresh['is_spike'] = metric_win_with_thresh['value'] > metric_win_with_thresh['global_P95']
metric_win_with_thresh['is_drop'] = metric_win_with_thresh['value'] <= metric_win_with_thresh['global_P15']
metric_win_with_thresh['is_anom'] = metric_win_with_thresh['is_spike'] | metric_win_with_thresh['is_drop']

# Per-pair window stats
metric_window_stats = (
    df_metric_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(max_value_in_window='max', min_value_in_window='min')
)

# Per-pair anomalous summary
metric_anom_group = (
    metric_win_with_thresh[metric_win_with_thresh['is_anom']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        anomalous_point_count_in_window=('is_anom','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        has_spike=('is_spike','any'),
        has_drop=('is_drop','any')
    )
)

metric_summary = (
    metric_anom_group
    .merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    .merge(metric_window_stats, on=['cmdb_id','kpi_name'], how='left')
)

def _severity_row(row):
    spike = np.nan
    drop = np.nan
    if row['has_spike'] and row['global_P95'] and row['global_P95']>0:
        spike = row['max_value_in_window'] / row['global_P95']
    if row['has_drop'] and row['min_value_in_window'] and row['min_value_in_window']>0:
        drop = row['global_P15'] / row['min_value_in_window']
    if np.isnan(spike) and np.isnan(drop):
        return np.nan
    if np.isnan(spike):
        return drop
    if np.isnan(drop):
        return spike
    return max(spike, drop)

metric_summary['severity_score'] = metric_summary.apply(_severity_row, axis=1)

# Select and format columns, round numerics
metric_summary = metric_summary[[
    'cmdb_id','kpi_name','global_P95','global_P15',
    'anomalous_point_count_in_window','earliest_anomaly_timestamp',
    'max_value_in_window','min_value_in_window','severity_score'
]]
for c in ['global_P95','global_P15','max_value_in_window','min_value_in_window','severity_score']:
    metric_summary[c] = pd.to_numeric(metric_summary[c], errors='coerce').round(6)

metric_summary_top = metric_summary.sort_values(
    by=['anomalous_point_count_in_window','severity_score'], ascending=[False, False]
).head(20)

# ---------------------------
# 2) trace.csv for candidates
# ---------------------------
# Select traces where cmdb_id in candidates OR trace_name contains candidate service substring
pattern = "|".join([s for s in candidates])
mask_trace = (df_trace['cmdb_id'].isin(candidates)) | df_trace['trace_name'].str.contains(pattern, na=False)
df_trace_candidates = df_trace[mask_trace].copy()

# Compute global thresholds per (cmdb_id, trace_name) over full series (use df_trace not filtered)
trace_thresholds = (
    df_trace[df_trace['trace_name'].str.contains("|".join([s.split('-service')[0] for s in candidates]), na=False) | df_trace['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P15=lambda x: x.quantile(0.15))
)

# Filter to incident window
df_trace_window = df_trace_candidates[
    (df_trace_candidates['timestamp'] >= inc_start) & (df_trace_candidates['timestamp'] <= inc_end)
].copy()

# Merge thresholds
trace_win_with_thresh = df_trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Mark anomalies
trace_win_with_thresh['is_spike'] = trace_win_with_thresh['value'] > trace_win_with_thresh['global_P95']
trace_win_with_thresh['is_drop'] = trace_win_with_thresh['value'] <= trace_win_with_thresh['global_P15']
trace_win_with_thresh['is_anom'] = trace_win_with_thresh['is_spike'] | trace_win_with_thresh['is_drop']

# Window stats
trace_window_stats = (
    df_trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(max_in_window='max', min_in_window='min')
)

# Anomalous summary
trace_anom_group = (
    trace_win_with_thresh[trace_win_with_thresh['is_anom']]
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        anomalous_count_in_window=('is_anom','size'),
        earliest_anomaly_timestamp=('timestamp','min'),
        has_spike=('is_spike','any'),
        has_drop=('is_drop','any')
    )
)

trace_summary = (
    trace_anom_group
    .merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    .merge(trace_window_stats, on=['cmdb_id','trace_name'], how='left')
)

def _severity_trace(row):
    spike = np.nan
    drop = np.nan
    if row['has_spike'] and row['global_P95'] and row['global_P95']>0:
        spike = row['max_in_window'] / row['global_P95']
    if row['has_drop'] and row['min_in_window'] and row['min_in_window']>0:
        drop = row['global_P15'] / row['min_in_window']
    if np.isnan(spike) and np.isnan(drop):
        return np.nan
    if np.isnan(spike):
        return drop
    if np.isnan(drop):
        return spike
    return max(spike, drop)

trace_summary['severity_score'] = trace_summary.apply(_severity_trace, axis=1)
trace_summary = trace_summary[[
    'trace_name','cmdb_id','global_P95','global_P15','anomalous_count_in_window',
    'earliest_anomaly_timestamp','max_in_window','min_in_window','severity_score'
]]
for c in ['global_P95','global_P15','max_in_window','min_in_window','severity_score']:
    trace_summary[c] = pd.to_numeric(trace_summary[c], errors='coerce').round(6)

trace_summary_top = trace_summary.sort_values(
    by=['anomalous_count_in_window','severity_score'], ascending=[False, False]
).head(20)

# ---------------------------
# 3) log.csv for candidates
# ---------------------------
df_log_candidates = df_log[df_log['cmdb_id'].isin(candidates)].copy()

# Compute global thresholds per (cmdb_id, log_name) over full series
log_thresholds = (
    df_log[df_log['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P15=lambda x: x.quantile(0.15))
)

# Filter to incident window
df_log_window = df_log_candidates[
    (df_log_candidates['timestamp'] >= inc_start) & (df_log_candidates['timestamp'] <= inc_end)
].copy()

# Merge thresholds
log_win_with_thresh = df_log_window.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')

# Mark anomalies
log_win_with_thresh['is_spike'] = log_win_with_thresh['value'] > log_win_with_thresh['global_P95']
log_win_with_thresh['is_drop'] = log_win_with_thresh['value'] <= log_win_with_thresh['global_P15']
log_win_with_thresh['is_anom'] = log_win_with_thresh['is_spike'] | log_win_with_thresh['is_drop']

# Window stats
log_window_stats = (
    df_log_window
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(max_in_window='max', min_in_window='min')
)

# Anomalous summary
log_anom_group = (
    log_win_with_thresh[log_win_with_thresh['is_anom']]
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(
        anomalous_count_in_window=('is_anom','size'),
        earliest_anomaly_timestamp=('timestamp','min')
    )
)

log_summary = (
    log_anom_group
    .merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    .merge(log_window_stats, on=['cmdb_id','log_name'], how='left')
)

# Format and round
log_summary = log_summary[[
    'cmdb_id','log_name','global_P95','global_P15','anomalous_count_in_window',
    'earliest_anomaly_timestamp','max_in_window','min_in_window'
]]
for c in ['global_P95','global_P15','max_in_window','min_in_window']:
    log_summary[c] = pd.to_numeric(log_summary[c], errors='coerce').round(6)

log_summary_top = log_summary.sort_values(
    by=['anomalous_count_in_window'], ascending=[False]
).head(20)

# Up to 10 raw log rows from the incident window for these services
log_raw_samples = df_log_window[['timestamp','cmdb_id','log_name','value']].sort_values('timestamp').head(10)

# ---------------------------
# 4) Combined raw anomalous metric and trace rows (for candidates) - up to 10 rows
# ---------------------------
# Metric anomalous rows for candidates (already computed metric_win_with_thresh)
metric_anomalous_rows = metric_win_with_thresh[metric_win_with_thresh['is_anom']][
    ['timestamp','cmdb_id','kpi_name','value']
].rename(columns={'kpi_name':'name'})

# Trace anomalous rows for candidates (trace_win_with_thresh)
trace_anomalous_rows = trace_win_with_thresh[trace_win_with_thresh['is_anom']][
    ['timestamp','cmdb_id','trace_name','value']
].rename(columns={'trace_name':'name'})

combined_anom = pd.concat([metric_anomalous_rows, trace_anomalous_rows], ignore_index=True)
combined_anom_sorted = combined_anom.sort_values('timestamp')[['timestamp','cmdb_id','name','value']].head(10)

# Final compact outputs (store full intermediate tables in variables above)
metric_summary_top, trace_summary_top, log_summary_top, log_raw_samples, combined_anom_sorted

```
Out[3]:
```
summary = (
    "Summary for candidate services (window 2024-01-24 02:06:40–02:36:40 UTC):\n\n"
    "Key observations:\n"
    "- Anomalies are concentrated around ~02:09–02:15 UTC across the five services.\n"
    "- Metrics show resource and latency changes, not application errors:\n"
    "  • Disk I/O: ts-order-service shows a notable diskio spike (max ~4.14e6 vs global P95 ~2.47e6; severity ~1.68).\n"
    "  • CPU: ts-travel-service, ts-order-service, ts-route-service and ts-train-service show CPU increases above their P95s (severity range ~1.09–1.64).\n"
    "  • Workload/throughput and latency: ts-order-service and ts-travel-service show workload and latency-50/90 anomalies.\n"
    "- Logs and traces indicate higher request volume but no error-rate increase:\n"
    "  • log.row_count for ts-auth, ts-order, ts-route, ts-travel show elevated counts at ~02:09 (e.g., ts-travel row_count up to 572; P95 ~521).\n"
    "  • trace error_rate series are zero in the sampled traces (no trace-level error spikes).\n"
    "  • error logs are absent (no error_log records for these services in the window).\n\n"
    "Interpretation:\n"
    "- The data points to a traffic/load spike around 02:09 UTC causing increased disk I/O and CPU usage and modest latency increases for several services (notably ts-order-service and ts-route-service for disk I/O, and ts-travel/ts-order/ts-route for CPU/latency).\n"
    "- There is no evidence of application errors or elevated trace error rates in this window; this looks like resource pressure from increased load rather than code failures.\n\n"
    "Recommended next steps:\n"
    "1) Investigate the traffic source at ~02:09 UTC (clients, batch jobs, or a deployment causing increased requests).\n"
    "2) Inspect disk I/O on ts-order-service and ts-route-service (possible I/O bottleneck) and CPU on ts-travel/ts-order/ts-route/ts-train.\n"
    "3) Correlate request traces (root→service spans) to find where latency accumulates downstream of the high-load services.\n"
    "4) Consider horizontal scaling or throttling for the high-load services if traffic spikes are expected.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_P95    global_P15  anomalous_point_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  severity_score
8    ts-order-service      diskio  2.471337e+06  46083.986288                                6  2024-01-24 02:09:00+00:00         4.144315e+06         41317.419295        1.676953
28  ts-travel-service         cpu  1.166293e+00      0.809467                                6  2024-01-24 02:12:00+00:00         1.910166e+00             0.788735        1.637810
7    ts-order-service         cpu  3.934503e+01      0.341049                                6  2024-01-24 02:13:00+00:00         3.969212e+01             0.245345        1.390082
14   ts-route-service         cpu  1.696304e+00      1.540801                                6  2024-01-24 02:15:00+00:00         2.160516e+00             1.392001        1.273661
13   ts-order-service    workload  6.991810e-01      0.244560                                6  2024-01-24 02:11:00+00:00         8.132000e-01             0.195650        1.249987
9    ts-order-service  latency-50  1.486020e-01      0.014279                                6  2024-01-24 02:10:00+00:00         1.637580e-01             0.011426        1.249706
21   ts-train-service         cpu  1.063245e+00      0.481451                                6  2024-01-24 02:11:00+00:00         1.270364e+00             0.456551        1.194799
17   ts-route-service  latency-90  7.612000e-03      0.005948                                6  2024-01-24 02:14:00+00:00         8.076000e-03             0.005182        1.147640
34  ts-travel-service    workload  1.828800e+00      1.492487                                6  2024-01-24 02:11:00+00:00         1.875550e+00             1.305500        1.143230
15   ts-route-service      diskio  5.978598e+04  43889.304672                                6  2024-01-24 02:09:00+00:00         6.363429e+04         38593.733351        1.137213
30  ts-travel-service  latency-50  5.003600e-02      0.034691                                6  2024-01-24 02:13:00+00:00         5.026600e-02             0.030828        1.125297
27   ts-train-service    workload  3.563320e+00      2.855597                                6  2024-01-24 02:12:00+00:00         3.633367e+00             2.578000        1.107679
10   ts-order-service  latency-90  2.323160e-01      0.022875                                6  2024-01-24 02:10:00+00:00         2.569140e-01             0.022477        1.105882
3     ts-auth-service  latency-90  8.583340e-01      0.482628                                6  2024-01-24 02:15:00+00:00         9.457230e-01             0.440127        1.101812
1     ts-auth-service      diskio  5.935859e+04  45959.922050                                6  2024-01-24 02:09:00+00:00         6.163166e+04         41743.532453        1.101007
22   ts-train-service      diskio  6.066880e+04  43365.182921                                6  2024-01-24 02:09:00+00:00         6.229368e+04         39738.380546        1.091267
12   ts-order-service      socket  2.367667e+01      8.663333                                6  2024-01-24 02:09:00+00:00         2.433333e+01             8.000000        1.082917
2     ts-auth-service  latency-50  2.764700e-01      0.214096                                6  2024-01-24 02:14:00+00:00         2.928250e-01             0.199523        1.073040
29  ts-travel-service      diskio  5.861088e+04  44157.340686                                6  2024-01-24 02:11:00+00:00         6.027470e+04         41316.448311        1.068759
5     ts-auth-service      socket  1.698000e+01     14.480000                                6  2024-01-24 02:16:00+00:00         1.733333e+01            13.716667        1.055650,                                   trace_name                  cmdb_id  global_P95  global_P15  anomalous_count_in_window earliest_anomaly_timestamp  max_in_window  min_in_window  severity_score
2        trace.to_ts-auth-service.error_rate                     root         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
6       trace.to_ts-order-service.error_rate                     root         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
10     trace.to_ts-travel-service.error_rate                     root         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
14     trace.to_ts-travel-service.error_rate  ts-admin-travel-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
18                trace.from_root.error_rate          ts-auth-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
22     trace.from_ts-auth-service.error_rate          ts-auth-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
26       trace.to_ts-auth-service.error_rate          ts-auth-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
30     trace.to_ts-travel-service.error_rate          ts-food-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
38                trace.from_root.error_rate         ts-order-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
46    trace.from_ts-order-service.error_rate         ts-order-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
50      trace.to_ts-order-service.error_rate         ts-order-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
54    trace.to_ts-station-service.error_rate         ts-order-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
58    trace.from_ts-route-service.error_rate         ts-route-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
62   trace.from_ts-travel-service.error_rate         ts-route-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
66  trace.from_ts-travel2-service.error_rate         ts-route-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
70      trace.to_ts-route-service.error_rate         ts-route-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
74    trace.from_ts-order-service.error_rate       ts-station-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
78   trace.from_ts-travel-service.error_rate    ts-ticketinfo-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
82    trace.from_ts-train-service.error_rate         ts-train-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN
86   trace.from_ts-travel-service.error_rate         ts-train-service         0.0         0.0                         25  2024-01-24 02:09:00+00:00            0.0            0.0             NaN,              cmdb_id         log_name  global_P95  global_P15  anomalous_count_in_window earliest_anomaly_timestamp  max_in_window  min_in_window
0    ts-auth-service  log.error_count         0.0         0.0                         25  2024-01-24 02:09:00+00:00              0              0
2   ts-order-service  log.error_count         0.0         0.0                         25  2024-01-24 02:09:00+00:00              0              0
6  ts-travel-service  log.error_count         0.0         0.0                         25  2024-01-24 02:09:00+00:00              0              0
4   ts-route-service  log.error_count         0.0         0.0                         25  2024-01-24 02:09:00+00:00              0              0
5   ts-route-service    log.row_count       361.2       310.0                          7  2024-01-24 02:09:00+00:00            373            124
1    ts-auth-service    log.row_count       381.6       334.4                          6  2024-01-24 02:09:00+00:00            386            128
3   ts-order-service    log.row_count        50.6        26.6                          6  2024-01-24 02:09:00+00:00             84              4
7  ts-travel-service    log.row_count       521.0       379.0                          6  2024-01-24 02:09:00+00:00            572            178,                     timestamp            cmdb_id         log_name  value
10  2024-01-24 02:09:00+00:00    ts-auth-service  log.error_count      0
11  2024-01-24 02:09:00+00:00    ts-auth-service    log.row_count    128
42  2024-01-24 02:09:00+00:00   ts-order-service  log.error_count      0
43  2024-01-24 02:09:00+00:00   ts-order-service    log.row_count      4
54  2024-01-24 02:09:00+00:00   ts-route-service  log.error_count      0
55  2024-01-24 02:09:00+00:00   ts-route-service    log.row_count    124
66  2024-01-24 02:09:00+00:00  ts-travel-service  log.error_count      0
67  2024-01-24 02:09:00+00:00  ts-travel-service    log.row_count    178
145 2024-01-24 02:10:00+00:00  ts-travel-service    log.row_count    444
144 2024-01-24 02:10:00+00:00  ts-travel-service  log.error_count      0,                     timestamp            cmdb_id                                      name         value
0   2024-01-24 02:09:00+00:00    ts-auth-service                                    diskio  44544.868159
268 2024-01-24 02:09:00+00:00  ts-travel-service                trace.from_root.error_rate      0.000000
267 2024-01-24 02:09:00+00:00  ts-travel-service             trace.from_root.duration_mean      0.031875
266 2024-01-24 02:09:00+00:00   ts-train-service       trace.to_ts-train-service.row_count    160.000000
265 2024-01-24 02:09:00+00:00   ts-train-service      trace.to_ts-train-service.error_rate      0.000000
264 2024-01-24 02:09:00+00:00   ts-train-service   trace.from_ts-travel2-service.row_count     20.000000
263 2024-01-24 02:09:00+00:00   ts-train-service  trace.from_ts-travel2-service.error_rate      0.000000
262 2024-01-24 02:09:00+00:00   ts-train-service    trace.from_ts-travel-service.row_count     20.000000
269 2024-01-24 02:09:00+00:00  ts-travel-service                 trace.from_root.row_count     10.000000
261 2024-01-24 02:09:00+00:00   ts-train-service   trace.from_ts-travel-service.error_rate      0.000000)```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing df_trace if available; otherwise load and parse
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp parsed
if df_trace['timestamp'].dtype != 'datetime64[ns, UTC]':
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Candidate service name substrings to match in trace_name or as cmdb_id
services = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-travel-service"]

# Incident window (UTC)
inc_start = pd.to_datetime("2024-01-24 02:06:40", utc=True)
inc_end = pd.to_datetime("2024-01-24 02:36:40", utc=True)

# Select rows where trace_name contains any of the service substrings OR cmdb_id is one of the services
pattern = "|".join(services)
mask_name = df_trace['trace_name'].str.contains(pattern, na=False)
mask_cmdb = df_trace['cmdb_id'].isin(services)
df_trace_selected = df_trace[mask_name | mask_cmdb].copy()

# 1) Compute global P95 (and P15) across the full series for these (trace_name, cmdb_id) pairs (store thresholds)
trace_thresholds = (
    df_trace_selected
    .groupby(['trace_name', 'cmdb_id'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95), global_P15=lambda x: x.quantile(0.15))
)

# 2) Filter selected traces to the incident window
df_trace_window = df_trace_selected[
    (df_trace_selected['timestamp'] >= inc_start) & (df_trace_selected['timestamp'] <= inc_end)
].copy()

# 3) Aggregate by (trace_name, cmdb_id) in the window
agg = (
    df_trace_window
    .groupby(['trace_name', 'cmdb_id'], as_index=False)
    .agg(
        count_in_window=('value', 'size'),
        earliest_timestamp_in_window=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        mean_value_in_window=('value', 'mean')
    )
)

# Attach previously computed global_P95 (and global_P15) if available
agg_with_thresh = agg.merge(trace_thresholds[['trace_name','cmdb_id','global_P95']], on=['trace_name','cmdb_id'], how='left')

# Round numeric columns for compact display
for c in ['global_P95','max_value_in_window','mean_value_in_window']:
    if c in agg_with_thresh.columns:
        agg_with_thresh[c] = pd.to_numeric(agg_with_thresh[c], errors='coerce').round(6)

# 1) Aggregated table limited to top 20 rows sorted by max_value_in_window descending
aggregated_table_top = agg_with_thresh.sort_values(by='max_value_in_window', ascending=False).head(20)

# 2) Up to 10 raw trace rows from the window for these trace_names sorted by value descending
raw_trace_samples = df_trace_window[['timestamp','cmdb_id','trace_name','value']].sort_values('value', ascending=False).head(10)

# Display compact results
aggregated_table_top, raw_trace_samples

```
Out[4]:
```
Summary of trace analysis (filtered to trace_name containing ts-auth-service, ts-order-service, ts-route-service, or ts-travel-service; window 2024-01-24 02:06:40–02:36:40 UTC):

Key aggregated findings (top by max_in_window):
- Highest observed row counts are for ts-route-service:
  - trace.from_ts-route-service.row_count and trace.to_ts-route-service.row_count (cmdb_id = ts-route-service)
  - count_in_window = 25, earliest = 2024-01-24 02:09:00 UTC
  - max_value_in_window = 746, mean ≈ 645.76, global_P95 = 722.4
  - Interpretation: route service experienced sustained high request volume, with peaks exceeding its historical P95 (746 > 722.4).

- ts-auth-service row counts:
  - trace.to_ts-auth-service.row_count and trace.from_ts-auth-service.row_count (cmdb_id = ts-auth-service)
  - count_in_window = 25, earliest = 02:09:00
  - max = 388, mean ≈ 339.68, global_P95 = 381.6
  - Interpretation: auth service also had elevated traffic, with peaks slightly above its P95.

- ts-travel-service row counts:
  - trace.from_ts-travel-service.row_count and trace.to_ts-travel-service.row_count (cmdb_id = ts-travel-service)
  - count_in_window = 25, earliest = 02:09:00
  - max = 365, mean ≈ 300.88, global_P95 = 350.6
  - Interpretation: travel service experienced elevated request volume (peaks above P95).

- Additional notable entries:
  - Several related row_count traces for ts-order-service and ts-train-service also show elevated counts (e.g., trace.to_ts-order-service.row_count max ≈ 78, global_P95 ≈ 46.4).
  - Many entries have earliest anomaly time around 02:09:00 UTC, indicating the load change began around that time and persisted across the window.

Top raw trace samples (highest values in the window, sorted by value):
- Multiple high row_count samples for ts-route-service (examples):
  - 2024-01-24 02:31:00 UTC — ts-route-service — trace.to_ts-route-service.row_count = 746
  - 2024-01-24 02:31:00 UTC — ts-route-service — trace.from_ts-route-service.row_count = 746
  - 2024-01-24 02:29:00 UTC — ts-route-service — trace.to_ts-route-service.row_count = 724
  - 2024-01-24 02:29:00 UTC — ts-route-service — trace.from_ts-route-service.row_count = 724
  - Several other timestamps (02:26, 02:11, 02:24) with values ~702–716.

Overall interpretation and recommendation:
- Traces show a clear traffic/row_count spike centered on ts-route-service (and elevated traffic to ts-auth-service and ts-travel-service). Many peak values exceed the computed global P95s, indicating abnormal high load.
- This points to a load/throughput event rather than increased error rates (error_rate traces were not elevated in prior checks).
- Recommended actions: investigate traffic sources (clients, jobs, or bursts) around 02:09 UTC, review routing to ts-route-service, and consider capacity/auto-scaling or request throttling to mitigate sustained high row counts.

The original code execution output of IPython Kernel is also provided below for reference:

(                                   trace_name                cmdb_id  count_in_window earliest_timestamp_in_window  max_value_in_window  mean_value_in_window  global_P95
39      trace.from_ts-route-service.row_count       ts-route-service               25    2024-01-24 02:09:00+00:00                746.0                645.76       722.4
89        trace.to_ts-route-service.row_count       ts-route-service               25    2024-01-24 02:09:00+00:00                746.0                645.76       722.4
67         trace.to_ts-auth-service.row_count        ts-auth-service               25    2024-01-24 02:09:00+00:00                388.0                339.68       381.6
19       trace.from_ts-auth-service.row_count        ts-auth-service               25    2024-01-24 02:09:00+00:00                388.0                339.68       381.6
55     trace.from_ts-travel-service.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                365.0                300.88       350.6
119      trace.to_ts-travel-service.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                365.0                300.88       350.6
59    trace.from_ts-travel2-service.row_count       ts-route-service               25    2024-01-24 02:09:00+00:00                228.0                200.56       221.8
91        trace.to_ts-route-service.row_count     ts-travel2-service               25    2024-01-24 02:09:00+00:00                228.0                200.56       221.8
90        trace.to_ts-route-service.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                159.0                122.32       143.6
52     trace.from_ts-travel-service.row_count       ts-route-service               25    2024-01-24 02:09:00+00:00                159.0                122.32       143.6
9                   trace.from_root.row_count        ts-auth-service               25    2024-01-24 02:09:00+00:00                 97.0                 84.92        95.4
66         trace.to_ts-auth-service.row_count                   root               25    2024-01-24 02:09:00+00:00                 97.0                 84.92        95.4
79        trace.to_ts-order-service.row_count       ts-order-service               25    2024-01-24 02:09:00+00:00                 78.0                 32.28        46.4
34      trace.from_ts-order-service.row_count       ts-order-service               25    2024-01-24 02:09:00+00:00                 78.0                 32.28        46.4
54     trace.from_ts-travel-service.row_count       ts-train-service               25    2024-01-24 02:09:00+00:00                 60.0                 45.20        55.0
103       trace.to_ts-train-service.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                 60.0                 45.20        55.0
99   trace.to_ts-ticketinfo-service.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                 58.0                 40.56        54.0
53     trace.from_ts-travel-service.row_count  ts-ticketinfo-service               25    2024-01-24 02:09:00+00:00                 58.0                 40.56        54.0
11                  trace.from_root.row_count      ts-travel-service               25    2024-01-24 02:09:00+00:00                 29.0                 20.44        27.0
116      trace.to_ts-travel-service.row_count                   root               25    2024-01-24 02:09:00+00:00                 29.0                 20.44        27.0,                      timestamp           cmdb_id                             trace_name  value
9023 2024-01-24 02:31:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  746.0
9011 2024-01-24 02:31:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  746.0
8223 2024-01-24 02:29:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  724.0
8211 2024-01-24 02:29:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  724.0
7019 2024-01-24 02:26:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  716.0
7031 2024-01-24 02:26:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  716.0
1047 2024-01-24 02:11:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  702.0
1035 2024-01-24 02:11:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  702.0
6219 2024-01-24 02:24:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  702.0
6231 2024-01-24 02:24:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  702.0)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing df_trace if present; otherwise load and parse timestamps
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Define target trace name substrings and incident window
targets = [
    'to_ts-route-service','from_ts-route-service',
    'to_ts-order-service','from_ts-order-service',
    'to_ts-travel-service','from_ts-travel-service',
    'to_ts-auth-service','from_ts-auth-service'
]
pattern = "|".join(targets)
inc_start = pd.to_datetime("2024-01-24 02:06:40", utc=True)
inc_end = pd.to_datetime("2024-01-24 02:36:40", utc=True)

# Select rows whose trace_name matches any target substring (use full df_trace for threshold calc)
mask_name_full = df_trace['trace_name'].str.contains(pattern, na=False)
df_trace_selected_full = df_trace[mask_name_full].copy()

# Compute global P95 for each (trace_name, cmdb_id) using the full available series (store thresholds)
trace_thresholds_specific = (
    df_trace_selected_full
    .groupby(['trace_name','cmdb_id'], as_index=False)['value']
    .agg(global_P95=lambda x: x.quantile(0.95))
)

# Filter to incident window (only rows of interest)
df_trace_window = df_trace_selected_full[
    (df_trace_selected_full['timestamp'] >= inc_start) & (df_trace_selected_full['timestamp'] <= inc_end)
].copy()

# Aggregate by (trace_name, cmdb_id) in the window
agg = (
    df_trace_window
    .groupby(['trace_name','cmdb_id'], as_index=False)
    .agg(
        count_in_window=('value','size'),
        earliest_timestamp_in_window=('timestamp','min'),
        max_value_in_window=('value','max'),
        mean_value_in_window=('value','mean')
    )
)

# Attach global_P95 where available
agg = agg.merge(trace_thresholds_specific[['trace_name','cmdb_id','global_P95']],
                on=['trace_name','cmdb_id'], how='left')

# Compute latency_severity for duration metrics (duration_mean or duration_p95)
is_duration = agg['trace_name'].str.contains('duration_mean|duration_p95', na=False)
agg['latency_severity'] = np.nan
mask = is_duration & agg['global_P95'].notna() & (agg['global_P95'] > 0)
agg.loc[mask, 'latency_severity'] = agg.loc[mask, 'max_value_in_window'] / agg.loc[mask, 'global_P95']

# Round numeric columns for compact display
for c in ['global_P95','max_value_in_window','mean_value_in_window','latency_severity']:
    if c in agg.columns:
        agg[c] = pd.to_numeric(agg[c], errors='coerce').round(6)

# 1) Aggregated table sorted by latency_severity desc (NaNs last), top 20
aggregated_table_top = agg.sort_values(by='latency_severity', ascending=False, na_position='last').head(20)

# 2) Up to 10 raw trace rows from window for these trace_names sorted by value desc
raw_trace_samples = df_trace_window[['timestamp','cmdb_id','trace_name','value']].sort_values('value', ascending=False).head(10)

# Display compact results
aggregated_table_top, raw_trace_samples

```
Out[5]:
```
summary = (
    "Trace analysis summary (window 2024-01-24 02:06:40–02:36:40 UTC):\n\n"
    "- Highest latency severity: trace.from_ts-travel-service.duration_p95 on ts-train-service (severity ≈ 1.64), meaning the peak duration exceeded its historical P95 by ~64%.\n"
    "- Other notable duration anomalies: to_ts-order-service.duration_mean (root and internal payment service) with severity ≈ 1.49–1.40, and ts-auth-service duration_p95/duration_mean with severity ≈ 1.30–1.15. Several duration_p95/duration_mean traces exceed their P95s by 10–50%.\n"
    "- Row-count (throughput) spikes are very large for ts-route-service (max values up to 746, mean ≈ 646, global P95 ≈ 722). ts-auth-service and ts-travel-service also show elevated row_count peaks above their P95s.\n"
    "- Raw top trace samples confirm repeated high row_count samples for ts-route-service at timestamps between ~02:11 and ~02:31 UTC (values ~702–746).\n\n"
    "Interpretation:\n"
    "- There is a concurrent pattern of heavy traffic (large row_count spikes, especially on ts-route-service) and increased inter-service latencies (notably travel→train, order-related spans, and auth spans).\n"
    "- This points to load-induced latency rather than isolated code errors, since error_rate traces were not elevated in prior checks.\n\n"
    "Recommended actions:\n"
    "1) Investigate traffic source and routing around 02:09 UTC that drove sustained high row counts on ts-route-service.\n"
    "2) Inspect capacity and resource usage for ts-route-service (hotspot for throughput) and for services showing latency severity (ts-train-service, ts-order-service, ts-auth-service).\n"
    "3) Correlate request traces end-to-end to find where latency accumulates and consider scaling or throttling to mitigate.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name                    cmdb_id  count_in_window earliest_timestamp_in_window  max_value_in_window  mean_value_in_window  global_P95  latency_severity
22   trace.from_ts-travel-service.duration_p95           ts-train-service               25    2024-01-24 02:09:00+00:00             0.010311              0.005038    0.006301          1.636364
40     trace.to_ts-order-service.duration_mean                       root               25    2024-01-24 02:09:00+00:00             0.184665              0.067911    0.123613          1.493893
41     trace.to_ts-order-service.duration_mean  ts-inside-payment-service               23    2024-01-24 02:10:00+00:00             0.130497              0.041137    0.093546          1.395006
1      trace.from_ts-auth-service.duration_p95            ts-auth-service               25    2024-01-24 02:09:00+00:00             0.023425              0.005406    0.017967          1.303781
35       trace.to_ts-auth-service.duration_p95            ts-auth-service               25    2024-01-24 02:09:00+00:00             0.023425              0.005406    0.017967          1.303781
18  trace.from_ts-travel-service.duration_mean           ts-train-service               25    2024-01-24 02:09:00+00:00             0.006023              0.003883    0.004877          1.235011
7     trace.from_ts-order-service.duration_p95         ts-station-service               25    2024-01-24 02:09:00+00:00             0.013240              0.008789    0.010805          1.225345
34       trace.to_ts-auth-service.duration_p95                       root               25    2024-01-24 02:09:00+00:00             1.369402              0.711907    1.131621          1.210123
33      trace.to_ts-auth-service.duration_mean            ts-auth-service               25    2024-01-24 02:09:00+00:00             0.005650              0.003154    0.004896          1.153972
0     trace.from_ts-auth-service.duration_mean            ts-auth-service               25    2024-01-24 02:09:00+00:00             0.005650              0.003154    0.004896          1.153972
70     trace.to_ts-travel-service.duration_p95            ts-food-service               25    2024-01-24 02:09:00+00:00             0.018061              0.013007    0.016010          1.128123
57      trace.to_ts-route-service.duration_p95         ts-travel2-service               25    2024-01-24 02:09:00+00:00             0.004719              0.003782    0.004253          1.109645
44      trace.to_ts-order-service.duration_p95  ts-inside-payment-service               23    2024-01-24 02:10:00+00:00             0.181076              0.061679    0.164682          1.099554
21   trace.from_ts-travel-service.duration_p95      ts-ticketinfo-service               25    2024-01-24 02:09:00+00:00             0.018181              0.013667    0.016609          1.094647
4    trace.from_ts-order-service.duration_mean           ts-order-service               25    2024-01-24 02:09:00+00:00             0.026444              0.009630    0.024534          1.077842
42     trace.to_ts-order-service.duration_mean           ts-order-service               25    2024-01-24 02:09:00+00:00             0.026444              0.009630    0.024534          1.077842
43      trace.to_ts-order-service.duration_p95                       root               25    2024-01-24 02:09:00+00:00             0.217390              0.095939    0.201939          1.076514
68     trace.to_ts-travel-service.duration_p95                       root               25    2024-01-24 02:09:00+00:00             0.071962              0.064259    0.067171          1.071329
45      trace.to_ts-order-service.duration_p95           ts-order-service               25    2024-01-24 02:09:00+00:00             0.103555              0.043231    0.097694          1.059997
6     trace.from_ts-order-service.duration_p95           ts-order-service               25    2024-01-24 02:09:00+00:00             0.103555              0.043231    0.097694          1.059997,                      timestamp           cmdb_id                             trace_name  value
9011 2024-01-24 02:31:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  746.0
9023 2024-01-24 02:31:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  746.0
8211 2024-01-24 02:29:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  724.0
8223 2024-01-24 02:29:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  724.0
7019 2024-01-24 02:26:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  716.0
7031 2024-01-24 02:26:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  716.0
6231 2024-01-24 02:24:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  702.0
6219 2024-01-24 02:24:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  702.0
1035 2024-01-24 02:11:00+00:00  ts-route-service  trace.from_ts-route-service.row_count  702.0
1047 2024-01-24 02:11:00+00:00  ts-route-service    trace.to_ts-route-service.row_count  702.0)```
```